In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# 1. Initialize data frames
passengers_day1 = [
    (101, "Rahul Sharma", "Hyderabad", "Economy", "India"),
    (102, "Priya Reddy", "Bangalore", "Business", "India"),
    (103, "Amit Kumar", "Mumbai", "Economy", "India"),
    (104, "Sneha Patel", "Delhi", "Premium Economy", "India"),
    (105, "Farhan Ali", "Chennai", "Economy", "India")
]
columns = ["passenger_id", "passenger_name", "city", "travel_class", "country"]
df_day1 = spark.createDataFrame(passengers_day1, columns)



In [0]:
# Define target DBFS path for the Delta Table
delta_path = "dbfs:/tmp/delta/passengers_delta"
dbutils.fs.rm(delta_path, recurse=True)


False

In [0]:
df_day1.write.format("delta").mode("overwrite").save(delta_path)




In [0]:
# 3. Verify record count
df_target = spark.read.format("delta").load(delta_path)
print(f"Day 1 Initial Target Row Count: {df_target.count()}")


Day 1 Initial Target Row Count: 5


In [0]:
# 4. Read and display Delta table
df_target.show()


+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:

# 5. View Delta history (Version 0 creation metadata)
dt = DeltaTable.forPath(spark, delta_path)
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-------------------+---------+------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                         |
+-------+-------------------+---------+------------------------------------------------------------+
|0      |2026-06-18 04:57:28|WRITE    |{mode -> Overwrite, statsOnLoad -> false, partitionBy -> []}|
+-------+-------------------+---------+------------------------------------------------------------+



In [0]:
passengers_day2 = [
    (102, "Priya Reddy", "Bangalore", "First Class", "India"),      
    (104, "Sneha Patel", "Hyderabad", "Premium Economy", "India"),  
    (106, "Neha Singh", "Pune", "Economy", "India"),               
    (107, "Arjun Verma", "Kochi", "Business", "India")              
]
df_day2 = spark.createDataFrame(passengers_day2, columns)

In [0]:
dt.alias("target").merge(
    source = df_day2.alias("source"),
    condition = "target.passenger_id = source.passenger_id"
).whenMatchedUpdate(set = {
    "passenger_name": "source.passenger_name",
    "city": "source.city",
    "travel_class": "source.travel_class",
    "country": "source.country"
}).whenNotMatchedInsert(values = {
    "passenger_id": "source.passenger_id",
    "passenger_name": "source.passenger_name",
    "city": "source.city",
    "travel_class": "source.travel_class",
    "country": "source.country"
}).execute()

print("--- Data Table After Upsert ---")
df_merged_result = spark.read.format("delta").load(delta_path)
df_merged_result.orderBy("passenger_id").show()

--- Data Table After Upsert ---
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
df_merged_result.filter(F.col("passenger_id") == 102).select("passenger_id", "travel_class").show()

+------------+------------+
|passenger_id|travel_class|
+------------+------------+
|         102| First Class|
+------------+------------+



In [0]:
df_merged_result.filter(F.col("passenger_id") == 106).show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
print("--- Version 0 (Day 1 State) ---")
df_v0.orderBy("passenger_id").show()

--- Version 0 (Day 1 State) ---
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
df_latest = spark.read.format("delta").load(delta_path)
print("--- Latest Version (Day 2 State) ---")
df_latest.orderBy("passenger_id").show()

--- Latest Version (Day 2 State) ---
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
print(f"Version 0 Count: {df_v0.count()} | Latest Version Count: {df_latest.count()}")

Version 0 Count: 5 | Latest Version Count: 7


In [0]:
print("Passenger 102 in V0:")
df_v0.filter(F.col("passenger_id") == 102).select("passenger_id", "travel_class").show()
print("Passenger 102 in Current Version:")
df_latest.filter(F.col("passenger_id") == 102).select("passenger_id", "travel_class").show()

Passenger 102 in V0:
+------------+------------+
|passenger_id|travel_class|
+------------+------------+
|         102|    Business|
+------------+------------+

Passenger 102 in Current Version:
+------------+------------+
|passenger_id|travel_class|
+------------+------------+
|         102| First Class|
+------------+------------+



In [0]:
print("Passenger 104 in V0:")
df_v0.filter(F.col("passenger_id") == 104).select("passenger_id", "city").show()
print("Passenger 104 in Current Version:")
df_latest.filter(F.col("passenger_id") == 104).select("passenger_id", "city").show()

Passenger 104 in V0:
+------------+-----+
|passenger_id| city|
+------------+-----+
|         104|Delhi|
+------------+-----+

Passenger 104 in Current Version:
+------------+---------+
|passenger_id|     city|
+------------+---------+
|         104|Hyderabad|
+------------+---------+



In [0]:
spark.sql(f"OPTIMIZE delta.`{delta_path}` ZORDER BY (city)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
dt.delete("passenger_id = 105")
print("--- Table after Deleting Passenger 105 ---")
spark.read.format("delta").load(delta_path).orderBy("passenger_id").show()

--- Table after Deleting Passenger 105 ---
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
dt.history().select("version", "operation", "operationParameters").show(truncate=False)

+-------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationParameters                                                                                                                                                                                                                         |
+-------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |OPTIMIZE |{clusterBy -> [], zOrderBy -> [], batchId -> 0, predicate -> [], auto -> true}                                                                                                                                    

In [0]:
dt.vacuum(0.0)
print("Vacuum execution completed cleanly!")